Positional Encoding

In [ ]:
import torch
from torch import nn

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=5000, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(max_len).unsqueeze(-1)
    formula = 10000 ** (torch.arange(0, d_model, 2).float() / d_model)
    pe[:, ::2] = torch.sin(position / formula)
    pe[:, 1::2] = torch.cos(position / formula)

    pe = pe.unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1)]
    return self.dropout(x)

Multi-head Attention

In [ ]:
class MultiheadAttention(nn.Module):
  def __init__(self, d_model, n_head, dropout):
    super().__init__()
    self.dropout = nn.Dropout(dropout)

    self.n_head = n_head
    self.d_model = d_model
    self.n_d_model = d_model // n_head

    self.w_q = nn.Linear(d_model, d_model)
    self.w_k = nn.Linear(d_model, d_model)
    self.w_v = nn.Linear(d_model, d_model)
    self.w_o = nn.Linear(d_model, d_model)

  def forward(self, q, k, v, mask=None):
    batch_size = q.size(0)
    q = self.w_q(q).view(batch_size, -1, self.n_head, self.n_d_model).transpose(1, 2)
    k = self.w_k(k).view(batch_size, -1, self.n_head, self.n_d_model).transpose(1, 2)
    v = self.w_v(v).view(batch_size, -1, self.n_head, self.n_d_model).transpose(1, 2) #(batch_size, n_head, len, n_d_model)

    scores = torch.matmul(q, k.transpose(-1, -2)) / (self.n_d_model ** 0.5) #(batch_size, n_head, len, len)
    if mask != None:
      mask = mask.unsqueeze(1).unsqueeze(2)
      scores = scores.masked_fill(mask == 0, -1e9)

    attention_weights = torch.softmax(scores, dim=-1)
    attention_weights = self.dropout(attention_weights)
    out = torch.matmul(attention_weights, v)
    out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

    return self.w_o(out)

Add&Norm

In [ ]:
class EncoderLayer(nn.Module):
  def __init__(self, d_model, d_ff, n_head, dropout):
    super().__init__()
    self.MultiheadAttention = MultiheadAttention(d_model, n_head, dropout)
    self.layernorm1 = nn.LayerNorm(d_model)
    self.layernorm2 = nn.LayerNorm(d_model)
    self.ffn = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(d_ff, d_model),
        nn.Dropout(dropout)
    )

  def forward(self, x, mask=None):
    attn_out = self.MultiheadAttention(x, x, x, mask)
    x = self.layernorm1(x + attn_out)
    ffn_out = self.ffn(x)
    x = self.layernorm2(x + ffn_out)
    return x

TransformerClassifier

In [ ]:
class TransformerClassifier(nn.Module):
  def __init__(self, vocab_size, max_len, d_model, n_head, d_ff, n_layers, num_class, dropout):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.positionalEncoding = PositionalEncoding(d_model, max_len, dropout)
    self.layers = nn.ModuleList([
        EncoderLayer(d_model, d_ff, n_head, dropout) for _ in range(n_layers)
    ])
    self.linear = nn.Linear(d_model, num_class)

  def forward(self, x, mask, is_mean_pooling=True):
    x = self.embedding(x)
    x = self.positionalEncoding(x)
    for layer in self.layers:
      x = layer(x, mask)

    if is_mean_pooling:
      x = x * mask.unsqueeze(-1).float()
      x = x.sum(dim=1)
      count_x = mask.sum(dim=-1).unsqueeze(-1).float()
      x = x / count_x
      return self.linear(x)

    else:
      cls_vector = x[:, 0, :]
      return self.linear(cls_vector)